# Run-time profile of the paper's likelihoods (referee comments 4 and 6)

Times every stage of the likelihoods the sampler calls, per evaluation inside a vmapped batch (the
sampler advances `num_delete = 100` chains together, so batch 100 is the operating point), on the GPU
and then on this VM's CPU cores (1, 4 and all cores, via `taskset`), plus JIT compile times:

* **GW170817** (mlgw_bns_jax, 128 s, 23–1500 Hz, H1L1V1): waveform at the 3199 relative-binning bin
  frequencies; RB at fixed (t_c, φ_c); RB + t_c marginalisation (2001 points); RB + t_c + φ_c
  marginalisation (**production**); full band at fixed (t_c, φ_c); full band + t_c + φ_c marginalisation;
  compile + first iteration of the production sampler. These split the production run's 20.6 min into
  waveform, RB overlaps, marginalisation and sampler overhead.
* **GW150914** (mlgw_bbh_jax): (2,2) on 4 s and all 7 modes on 8 s, waveform and full-band likelihood.

Needs the GW170817 frozen data on Drive (`MyDrive/gwjax_referee/gw170817_data/data_gwtc1/`, the same
folder the GW170817 notebook reads); the GW150914 inputs are fetched from GitHub.
Results go to `MyDrive/gwjax_referee/profiling/<timestamp>/`.

**Order:** run the install cell, restart the session, then run everything from the top.
Expect about 1 h in total, most of it in the 1-core CPU pass.

## 1. Install

In [ ]:
import importlib.util, os

RUNNING_ON_COLAB  = importlib.util.find_spec("google.colab") is not None
ALREADY_INSTALLED = importlib.util.find_spec("gwgpu_jax") is not None

if RUNNING_ON_COLAB and not ALREADY_INSTALLED:
    # Use Colab's own GPU jax: its CUDA plugin matches the image's CUDA/cuDNN (forcing jax 0.4.31
    # over it crashed the kernel on the Python 3.12 image). Install jax only if it is missing or has
    # no GPU, then freeze jax/TF so the gwgpu_jax install below cannot change them.
    import subprocess as _sp, sys as _sys
    if _sp.run([_sys.executable, "-c", "import jax, sys; sys.exit(jax.default_backend() != 'gpu')"],
               capture_output=True).returncode != 0:
        !pip install -q -U "jax[cuda12]"
    !pip freeze | grep -iE "^(jax|jaxlib|jax-cuda[0-9]+-(plugin|pjrt)|tensorflow|keras|ml.dtypes|numpy)==" > /content/gwjax_constraints.txt
    !python -c "import jax; print('jax', jax.__version__, jax.devices())"

    # Public repo - no token, no authentication. Installs from the default branch.
    !pip install -q -c /content/gwjax_constraints.txt "gwgpu_jax[data,mlgw] @ git+https://github.com/saulo-albuquerque-phys/GWgpu-jax.git@py312_modern_jax"
    # tf2jax <= 0.3.7 uses jax.util.safe_zip (removed in jax 0.7); 0.3.8 is the jax>=0.7 release.
    # It declares TF >= 2.20 but does not check it at run time -> install without deps (Colab TF is
    # frozen); the waveform self-check cell below validates the converted networks.
    if _sp.run([_sys.executable, "-c", "import jax, sys; sys.exit(tuple(map(int, jax.__version__.split('.')[:2])) < (0, 7))"]).returncode == 0:
        !pip install -q --no-deps "tf2jax==0.3.8"

    print("Installed. If JAX was re-pinned: Runtime -> Restart session, then re-run from here.")
else:
    print("gwgpu_jax already importable - skipping install.")


In [ ]:
# ── Vendored-subrepo bandaid for `gwgpu_jax[mlgw]` on Colab ────────────
# The `mlgw_bbh_jax` Python package is a vendored sub-repo that lives at
#     <gwgpu_jax-install>/mlgw_jax/mlgw_bbh_jax/
# but it's not committed to the GWgpu_jax repo Colab clones from, so the
# wheel built on Colab is missing it. We clone it from the upstream
# `gwgpu_jax_compatible` branch of adrianomascioli/MLGW-JAX, which already
# bakes in the inference-compatibility fixes (Keras 2/3 model-load compat,
# optional keras_tuner / GW_helper imports, JAX-traceable Wigner-d), so
# — unlike the original notebook — NO on-the-fly patch is applied.
#
# Once `mlgw_bbh_jax` is properly committed to the GWgpu_jax repo (or pinned
# as a submodule), this whole cell can be deleted.
import subprocess, sys, shutil
from pathlib import Path

# FIX: Downgrade cryptography to fix pyOpenSSL incompatibility (module 'lib' has no attribute 'GEN_EMAIL')
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "cryptography<44", "pyopenssl"],
    check=True,
)

# 1. Install training-only deps the upstream sub-repo may import. On the
#    gwgpu_jax_compatible branch these imports are already guarded, so this
#    is belt-and-braces and can be dropped if you like.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "keras-tuner"],
    check=True,
)

# 2. Clone the vendored sub-repo (gwgpu_jax_compatible branch) into the
#    gwgpu_jax install path.
import gwgpu_jax
_GW    = Path(gwgpu_jax.__file__).resolve().parent
_DEST  = _GW / "mlgw_jax" / "mlgw_bbh_jax"
_READY = _DEST / ".gwgpu_jax_compatible_ready"

if _READY.exists():
    print(f"✓ mlgw_bbh_jax (gwgpu_jax_compatible) already present at {_DEST}")
else:
    if _DEST.exists():
        shutil.rmtree(_DEST)
    print(f"Cloning mlgw_bbh_jax (gwgpu_jax_compatible) into {_DEST} (~1 GB, ~1–2 min on Colab)…")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", "gwgpu_jax_compatible",
        "https://github.com/adrianomascioli/MLGW-JAX.git",
        str(_DEST),
    ], check=True)
    _READY.touch()
    print(f"✓ mlgw_bbh_jax cloned (gwgpu_jax_compatible) at {_DEST}")

# 3. Verify the wrapper can now import the vendored Python package.
from gwgpu_jax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator  # noqa: F401
print("✓ MLGWBBHGenerator importable")

In [ ]:
# Waveform self-check: mlgw_bbh_jax model_4, all 7 modes, vs reference values from the validated environment
# (python 3.11.15, jax 0.4.31, tensorflow 2.15.1, keras 2.15.0). Guards against jax / TF / Keras / tf2jax version changes on Colab.
import numpy as _np, jax as _jax, jax.numpy as _jnp
_jax.config.update("jax_enable_x64", True)
from gwgpu_jax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator as _G
_REF = {"case0":{"params":[36.0,29.0,0.1,-0.2],"hp":[2.684232668750469e-22,-3.15298047963779e-22,-5.178147261774714e-22,1.2028925313823429e-22,1.2085156389122561e-22,8.74947157607198e-22,-1.1061629153576286e-21,-1.4269387091847638e-21,8.889210769879484e-22,-1.2372876202635734e-21,-1.350760460057205e-21,1.0699164249469722e-22],"hc":[-2.674116771603285e-22,3.0788161571252814e-22,8.118996257996721e-24,5.87820093995615e-22,6.941320671055936e-22,1.9682902862941935e-22,-3.5499824442357526e-22,2.160667846587019e-22,-1.448541147603001e-21,-1.2926729321450586e-21,1.1422242259615139e-21,8.0379300079139075e-22]},"case1":{"params":[60.0,10.0,0.5,0.3],"hp":[-2.4734425172214526e-22,6.914309383883661e-23,1.877723818510623e-22,2.6753268700968177e-22,3.899698816822955e-22,5.269218466268872e-22,7.411386293791411e-23,6.691837162797455e-22,-4.254461031478611e-22,6.421692902616687e-22,9.501365175363906e-22,2.6111431811907688e-22],"hc":[9.335571625628856e-23,-2.932173446149296e-22,2.921996051510419e-22,3.0170348226358025e-22,2.291729169835735e-22,2.2413740818464353e-22,6.534263458022117e-22,-2.6652034630447447e-22,7.655800630577663e-22,7.402441288891662e-22,-5.491785440161921e-22,-2.6442400397555953e-22]},"idx":[8000,12000,14000,15000,15500,15800,15900,15950,15970,15975,15980,16000]}
_g = _G(model=4)
_t = _jnp.linspace(-3.9, 0.1, 16384, endpoint=False)
for _k in ("case0", "case1"):
    _hp, _hc = _g.get_hphc(_t, *_REF[_k]["params"], 400.0, 0.5, 0.3, modes=None)
    for _name, _h in (("hp", _hp), ("hc", _hc)):
        _r = _np.array(_REF[_k][_name])
        _err = float(_np.max(_np.abs(_np.asarray(_h)[_REF["idx"]] - _r)) / _np.max(_np.abs(_r)))
        assert _err < 1e-6, f"mlgw_bbh {_k} {_name} differs from reference: rel. diff {_err:.1e}"
import tensorflow as _tf
print(f"✓ mlgw_bbh 7-mode waveforms match the reference (jax {_jax.__version__}, TF {_tf.__version__})")
del _g


## 2. Inputs, scripts and machine

In [ ]:
import os, json, subprocess, datetime, urllib.request
from pathlib import Path
RAW = "https://raw.githubusercontent.com/saulo-albuquerque-phys/GWgpu-jax/py312_modern_jax/examples/referee_response"
WORK = Path("/content/profiling"); WORK.mkdir(exist_ok=True)
for rel in ("profiling/profile_pe.py", "profiling/summarise_profile.py",
            "gw150914_mlgw_bbh/data_bilby_hm_8s/gw150914_data.npz",
            *[f"gw150914_mlgw_bbh/data_lvk_welch/{n}" for n in ("metadata.json", "frequency_array.npy", "H1_psd.npy",
                                                               "H1_strain_fd.npy", "L1_psd.npy", "L1_strain_fd.npy")]):
    dst = WORK / rel; dst.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{RAW}/{rel}", dst)

from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
GW170817_DATA = "/content/drive/MyDrive/gwjax_referee/gw170817_data/data_gwtc1"
assert Path(GW170817_DATA, "metadata.json").exists(), f"copy data_gwtc1/ to {GW170817_DATA} first"
OUT_DIR = f"/content/drive/MyDrive/gwjax_referee/profiling/{datetime.datetime.now():%Y%m%d_%H%M%S}"
os.makedirs(OUT_DIR, exist_ok=True)

N_CPU = len(os.sched_getaffinity(0))
machine = dict(gpu=subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                  capture_output=True, text=True).stdout.strip(),
               cpu=next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")),
               cpu_cores=N_CPU, ram_gb=round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))
json.dump(machine, open(f"{OUT_DIR}/machine.json", "w"), indent=2)
print(machine); print("results ->", OUT_DIR)
COMMON = (f"--gw170817-data {GW170817_DATA} --gw150914-data {WORK}/gw150914_mlgw_bbh/data_lvk_welch "
          f"--gw150914-hm-data {WORK}/gw150914_mlgw_bbh/data_bilby_hm_8s/gw150914_data.npz")
FILTER = 'grep -vE "features_spec|ListWrapper|keras|^ *$|oneDNN|cuda_|TF-TRT|tensorflow|Warning"'

## 3. Acceptance-walk kernel (for the sampler compile-time measurement)

In [ ]:
import os, sys, subprocess
KDIR = "/content/blackjax_ns_gw"
if not os.path.isdir(f"{KDIR}/src/custom_kernels"):
    subprocess.run(["git","clone","--no-checkout","--depth","1","--filter=blob:none",
                    "https://github.com/mrosep/blackjax_ns_gw.git", KDIR], check=True)
    subprocess.run(["git","-C",KDIR,"sparse-checkout","set","src/custom_kernels"], check=True)
    subprocess.run(["git","-C",KDIR,"checkout"], check=True)
sys.path.insert(0, f"{KDIR}/src")
os.environ["GWJAX_ACCEPTANCE_WALK_SRC"] = f"{KDIR}/src"
import custom_kernels
print("✓ acceptance-walk kernel ready:", custom_kernels.__file__)

## 4. GPU profile

Batches 1, 10, 100, 1000; GW170817 also times the production sampler (1000 live points) for one and
three iterations, which separates its compile + setup time from the per-iteration time.

In [ ]:
!cd {WORK} && python -u profiling/profile_pe.py --what gw170817 gw150914_22 gw150914_hm --batches 1 10 100 1000 \
    --sampler-compile --label "GPU {machine['gpu'].split(',')[0]}" --out {OUT_DIR}/profile_gpu.json {COMMON} 2>&1 | {FILTER}

## 5. CPU profiles: 1, 4 and all cores

Same code on this VM's CPU (`JAX_PLATFORMS=cpu`, GPU hidden), restricted to the first N cores with
`taskset` (XLA's CPU thread pool sizes itself from the cores it is allowed to use). Batches 1, 10, 100.
The full-band t_c/φ_c-marginalised GW170817 likelihood is skipped here: it takes ~25 min to compile on a CPU
(it is only there to show what the run would cost without relative binning, which the GPU pass covers).
This is the batched multi-core CPU baseline of comment 4.

In [ ]:
for n in sorted({1, min(4, N_CPU), N_CPU}):
    print(f"\n######## CPU, {n} core(s) ########")
    !cd {WORK} && CUDA_VISIBLE_DEVICES="" JAX_PLATFORMS=cpu taskset -c 0-{n-1} python -u profiling/profile_pe.py \
        --what gw170817 gw150914_22 gw150914_hm --batches 1 10 100 --skip-full-marg --label "CPU {n} core(s)" \
        --out {OUT_DIR}/profile_cpu_{n}.json {COMMON} 2>&1 | {FILTER}

## 6. Summary tables

Per-evaluation cost at batch 100 on every device, compile times, and the split of the GW170817
production run (7.46e6 likelihood calls in 1236 s on an A100) into waveform, RB overlaps,
t_c/φ_c marginalisation and sampler overhead. Saved as `profile_summary.json`.

In [ ]:
import glob
files = sorted((f for f in glob.glob(f"{OUT_DIR}/profile_*.json") if not f.endswith("_summary.json")), key=lambda f: ("cpu" in f, len(f), f))
!cd {WORK} && python profiling/summarise_profile.py {" ".join(files)} --batch 100 --out {OUT_DIR}/profile_summary.json
print("\nall results in", OUT_DIR)